# Challenge One: Real-Time Weather Alerts Agent

**Goal:** Demonstrate the ability to create and test an agent using the Google Agent Development Kit (ADK).

**Requirements covered in this notebook:**
1. An agent that uses tools to retrieve real-time weather data for user locations.
2. A weather summary / alert based on current conditions and location.
3. Test code demonstrating the agent works for multiple US cities.
4. Support for both a Gemini model and a third-party model (OpenAI GPT via LiteLLM).

Author: Akhil Sharma (WWT)


In [ ]:
# 1. Install dependencies
!pip install --upgrade --quiet google-adk google-cloud-aiplatform litellm requests openai


In [ ]:
# 2. Imports and configuration
import os
import requests
from typing import Optional, List, Dict, Tuple

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

# --- Configuration ---
# Set these as environment variables (recommended) or fill in directly.
# GOOGLE_MAPS_API_KEY: Google Maps Platform API key with the Geocoding API enabled.
# GOOGLE_API_KEY / GOOGLE_CLOUD_PROJECT: used by ADK/Vertex for the Gemini model.
# OPENAI_API_KEY: used by LiteLLM to call GPT as the third-party model.

GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY", "YOUR_GOOGLE_MAPS_API_KEY")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY  # always apply the latest value (avoid stale setdefault)

MODEL_GEMINI_FLASH = "gemini-2.5-flash"
MODEL_GPT = "openai/gpt-4o"


In [ ]:
# 3. Tool: convert a place name to latitude/longitude using the Google Maps Geocoding API
def get_lat_lon(place: str) -> Optional[Tuple[float, float]]:
    """
    Convert a place name (e.g. a city and state) into geographic coordinates
    using the Google Maps Geocoding API.

    Args:
        place (str): A human-readable location, e.g. "Chicago, IL" or
            "1600 Amphitheatre Parkway, Mountain View, CA".

    Returns:
        Optional[Tuple[float, float]]: A (latitude, longitude) tuple, or
        None if the location could not be geocoded or an error occurred.
    """
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": place, "key": GOOGLE_MAPS_API_KEY}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        if data.get("status") != "OK" or not data.get("results"):
            return None

        location = data["results"][0]["geometry"]["location"]
        return (location["lat"], location["lng"])
    except (requests.RequestException, KeyError, IndexError):
        return None


In [ ]:
# 4. Tool: fetch the extended weather forecast from the National Weather Service API
def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """
    Fetch the extended weather forecast from the U.S. National Weather Service API
    based on a given latitude and longitude.

    Args:
        lat (float): Latitude of the location (e.g., 38.8977).
        lon (float): Longitude of the location (e.g., -77.0365).

    Returns:
        Optional[List[Dict[str, str]]]: A list of forecast period dictionaries,
        each containing keys such as "name", "temperature", "temperatureUnit",
        "windSpeed", "windDirection", "shortForecast", and "detailedForecast".
        Returns None if data is unavailable or an error occurs.
    """
    headers = {"User-Agent": "adk-weather-alerts-agent (contact: akhil.sharma@wwt.com)"}

    try:
        # Step 1: resolve the lat/lon to a NWS gridpoint / forecast URL.
        points_url = f"https://api.weather.gov/points/{lat},{lon}"
        points_resp = requests.get(points_url, headers=headers, timeout=10)
        points_resp.raise_for_status()
        forecast_url = points_resp.json()["properties"]["forecast"]

        # Step 2: fetch the extended forecast periods.
        forecast_resp = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_resp.raise_for_status()
        periods = forecast_resp.json()["properties"]["periods"]

        return [
            {
                "name": p.get("name", ""),
                "temperature": str(p.get("temperature", "")),
                "temperatureUnit": p.get("temperatureUnit", ""),
                "windSpeed": p.get("windSpeed", ""),
                "windDirection": p.get("windDirection", ""),
                "shortForecast": p.get("shortForecast", ""),
                "detailedForecast": p.get("detailedForecast", ""),
            }
            for p in periods
        ]
    except (requests.RequestException, KeyError, IndexError):
        return None


## Building the agent

The agent below is given both tools (`get_lat_lon` and `get_extended_weather_forecast`) and
instructions that tell it how to combine them: geocode the requested place, pull the forecast,
then summarize current conditions and flag anything alert-worthy (severe heat/cold, high wind,
storms, etc.).


In [ ]:
# 5. Agent instructions
WEATHER_AGENT_INSTRUCTIONS = """
You are Pat, a friendly and knowledgeable weather alerts assistant for locations in the
United States.

When a user asks about the weather for a place:
1. Use the `get_lat_lon` tool to convert the place name into latitude/longitude.
   If it fails, tell the user you could not find that location and ask them to clarify
   (e.g. add a state).
2. Use the `get_extended_weather_forecast` tool with those coordinates to retrieve the
   forecast periods.
3. Summarize the current/upcoming conditions in plain language: temperature, wind, and
   general outlook.
4. Proactively call out anything alert-worthy: extreme heat (>= 95F) or cold (<= 20F),
   high winds (>= 25 mph), or forecasts mentioning storms, tornadoes, snow, or ice. If
   nothing stands out, say conditions look normal.
5. Keep responses concise and easy to scan, and always name the city/location you are
   reporting on.
"""


In [ ]:
# 6. Agent variant 1: Gemini model
weather_agent_gemini = Agent(
    name="Pat",
    model=MODEL_GEMINI_FLASH,
    description="Pat the Friendly Weather Agent (Gemini).",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_extended_weather_forecast, get_lat_lon],
)

# 7. Agent variant 2: third-party model (OpenAI GPT) via LiteLLM
weather_agent_gpt = Agent(
    name="PatGPT",
    model=LiteLlm(model=MODEL_GPT),
    description="Pat the Friendly Weather Agent (GPT).",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_extended_weather_forecast, get_lat_lon],
)


## Running the agents

Each agent is hosted in a small `AdkApp`, given a session, and then queried. The helper
function below wraps the three steps shown in the workshop slides (create app -> create
session -> query) so we can reuse it for every city and every model.


In [ ]:
# 8. Helper to run a query against an agent and print the response
from vertexai.preview import reasoning_engines
from IPython.display import Markdown, display

def ask_agent(agent: Agent, question: str, user_id: str = "test-user-id") -> str:
    """
    Host the given agent in an AdkApp, create a session, and query it once.

    Args:
        agent (Agent): The ADK agent to run.
        question (str): The natural-language question/prompt to send.
        user_id (str): An identifier for the querying user/session owner.

    Returns:
        str: The text of the agent's final response.
    """
    app = reasoning_engines.AdkApp(agent=agent)
    session = app.create_session(user_id=user_id)

    # Depending on the installed google-cloud-aiplatform version,
    # create_session() returns either an object with an `.id` attribute
    # or a plain dict with an "id" key. Handle both.
    session_id = session["id"] if isinstance(session, dict) else session.id

    last_event = None
    try:
        for event in app.stream_query(
            user_id=user_id,
            session_id=session_id,
            message=question,
        ):
            last_event = event
    except Exception as e:
        return f"Error while querying agent '{agent.name}': {e}"

    # A failed model/tool call sometimes surfaces as an event without a
    # "content" key (e.g. a rate-limit or auth error) rather than a raised
    # exception. Surface that clearly instead of crashing on a KeyError.
    if not last_event or "content" not in last_event:
        return f"Agent '{agent.name}' did not return a valid response. Raw event: {last_event}"

    return last_event["content"]["parts"][0]["text"]


In [ ]:
# 9. Test code: run the Gemini-backed agent against multiple US cities
test_cities = [
    "New York, NY",
    "Los Angeles, CA",
    "Chicago, IL",
    "Miami, FL",
    "Seattle, WA",
]

print("=== Gemini agent ===")
for city in test_cities:
    print(f"\n--- {city} ---")
    response = ask_agent(weather_agent_gemini, f"What's the weather like in {city}? Any alerts I should know about?")
    display(Markdown(response))


In [ ]:
# 10. Test code: run the GPT-backed agent against the same cities to confirm
# the agent also works with a non-Gemini, third-party model.
print("=== GPT agent (via LiteLLM) ===")
for city in test_cities:
    print(f"\n--- {city} ---")
    response = ask_agent(weather_agent_gpt, f"What's the weather like in {city}? Any alerts I should know about?")
    display(Markdown(response))


## Notes

- Replace the placeholder API keys in the configuration cell with real values before running
  (Google Maps Geocoding API key, and an OpenAI API key for the GPT variant). In Colab
  Enterprise these can be pulled from Secret Manager or set as notebook secrets instead of
  hardcoding.
- The National Weather Service API only covers US locations and does not require an API key.
- `get_lat_lon` and `get_extended_weather_forecast` are plain, type-hinted Python functions with
  docstrings, so the ADK can pass them to the agent as callable tools directly.
